<a href="https://colab.research.google.com/github/Kishoby/Final-Research/blob/Hybrid-model/Online_Incremental_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install river pandas==2.2.3


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 1.1 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.


In [ ]:
import pandas as pd
from river import preprocessing, linear_model, metrics
from collections import deque


In [ ]:
df1 = pd.read_csv("/content/drive/MyDrive/Research/COMURS/04 09 2025 to 06 08 2025.csv")
df2 = pd.read_csv("/content/drive/MyDrive/Research/COMURS/06 09 2025 to 08 08 2025.csv")
df3 = pd.read_csv("/content/drive/MyDrive/Research/COMURS/08 09 2025 to 10 08 2025.csv")


In [ ]:
FEATURES = [
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index",
    "air_quality_PM10",
    "air_quality_Carbon_Monoxide",
    "air_quality_Nitrogen_dioxide",
    "air_quality_Sulphur_dioxide",
    "humidity",
    "cloud",
    "visibility_miles",
    "visibility_km",
    "longitude",
    "gust_kph",
    "precip_mm",
    "precip_in",
    "air_quality_Ozone"
]

TARGET = "air_quality_PM2.5"


In [ ]:
model = preprocessing.StandardScaler() | linear_model.LinearRegression()


In [ ]:
mse  = metrics.MSE()
rmse = metrics.RMSE()
mae  = metrics.MAE()
r2   = metrics.R2()
mape = metrics.MAPE()


In [ ]:
WINDOW_SIZE = 200   # reduce for testing if dataset is small
buffer_X = deque(maxlen=WINDOW_SIZE)
buffer_y = deque(maxlen=WINDOW_SIZE)


In [ ]:
def hybrid_learning(df, iteration_name):
    print(f"\n🔁 {iteration_name}")

    for _, row in df.iterrows():

        # -------- Feature dictionary (online)
        x = {f: row[f] for f in FEATURES}
        y = row[TARGET]

        # -------- ONLINE LEARNING (short-term)
        y_pred = model.predict_one(x)
        model.learn_one(x, y)

        # -------- METRICS UPDATE
        mse.update(y, y_pred)
        rmse.update(y, y_pred)
        mae.update(y, y_pred)
        r2.update(y, y_pred)
        mape.update(y, y_pred)

        # -------- STORE FOR INCREMENTAL UPDATE
        buffer_X.append(x)
        buffer_y.append(y)

        # -------- INCREMENTAL LEARNING (batch)
        if len(buffer_X) == WINDOW_SIZE:
            X_batch = pd.DataFrame(buffer_X)
            y_batch = pd.Series(buffer_y)

            model.learn_many(X_batch, y_batch)

            buffer_X.clear()
            buffer_y.clear()

    # -------- PRINT METRICS
    print("MSE  :", mse.get())
    print("RMSE :", rmse.get())
    print("MAE  :", mae.get())
    print("R²   :", r2.get())
    print("MAPE :", mape.get())


In [ ]:
hybrid_learning(df1, "Iteration 1 – Stream 1")
hybrid_learning(df2, "Iteration 2 – Stream 2")
hybrid_learning(df3, "Iteration 3 – Stream 3")



🔁 Iteration 1 – Stream 1
MSE  : 31094.71080660944
RMSE : 176.33692411576607
MAE  : 36.03998368811698
R²   : -23.787576756663878
MAPE : 279.4678917546217

🔁 Iteration 2 – Stream 2
MSE  : 15883.223081101642
RMSE : 126.02865976079266
MAE  : 22.765471952574202
R²   : -12.740778011418318
MAPE : 170.4344299121555

🔁 Iteration 3 – Stream 3
MSE  : 33448.519282337234
RMSE : 182.88936350246624
MAE  : 37.88496335363587
R²   : -29.111225366880323
MAPE : 245.8034566317929
